## CGM + time_since_meal + image (vit, residual fusion, dim=4)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [5]:
import json

import numpy as np
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

MODEL_NAME = 'CGM+time+vit (residual, image-only, dim=4)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

ENCODER_NAME = 'vit'
IMG_FEAT_DIM = 768
IMG_PROJ_DIM = 4

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '5_residual_multimodal'
EXP2_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'residual_vit'

SUMMARY_PATH = RESULTS_DIR / 'residual_vit_dim4_validation_summary.json'
PAIRED_PATH = RESULTS_DIR / 'residual_vit_dim4_paired_validation.csv'

print('MODEL_NAME:', MODEL_NAME)
print('IMG_PROJ_DIM:', IMG_PROJ_DIM)

MODEL_NAME: CGM+time+vit (residual, image-only, dim=4)
IMG_PROJ_DIM: 4


In [6]:
stats = data.load_cgm_stats()

train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)

print('n_train_batches:', len(train_loader))
print('n_val_batches:', len(val_loader))

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])

n_train_batches: 323
n_val_batches: 64
batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [7]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\n{'-' * 18} seed={seed} {'-' * 18}")

    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)
    train.set_seed(seed)

    model = FrozenImageOnlyResidualPredictor(base_model=base_model, img_feat_dim=IMG_FEAT_DIM, img_proj_dim=IMG_PROJ_DIM)

    baseline_trainable = [
        name for name, p in model.named_parameters()
        if p.requires_grad and (name.startswith('gru.') or name.startswith('time_proj.') or name.startswith('base_head.'))
    ]
    if baseline_trainable:
        raise RuntimeError(' baseline is not fully frozen. '
                           f'Unexpected trainable parameters: {baseline_trainable}')

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    print('trainable parameters:')
    for name in trainable_names:
        print('  ', name)

    # Zero-initialised residual
    check_batch = next(iter(val_loader))
    cgm, tsm, has_meal, img_feat, _ = check_batch

    model.eval()
    base_model.eval()

    with torch.no_grad():
        base_pred = base_model(cgm, tsm, has_meal)
        residual_pred = model(cgm, tsm, img_feat, has_meal)

    init_diff = (base_pred - residual_pred).abs().max().item()
    print(f'initial max |residual - baseline| = {init_diff:.8f}')

    if init_diff > 1e-5:
        raise RuntimeError(f'seed={seed}: residual model does not match the frozen baseline '
        f'at initialisation (max diff={init_diff}).')

    model, history = train.train_model(model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'], lr=1e-4, weight_decay=1e-5, forward_fn=forward_fn,
        checkpoint_dir=(EXPERIMENT_DIR / 'checkpoints'/ 'residual_vit' /f'residual_vit_dim4_seed{seed}'), verbose=False, progress_bar=True, progress_desc=f'seed={seed}')

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'encoder': ENCODER_NAME,
        'img_feat_dim': IMG_FEAT_DIM,
        'img_proj_dim': IMG_PROJ_DIM,
        'best_epoch': len(history['train_loss']),
        'initial_max_abs_diff_vs_baseline': init_diff,
        'val_results': val_results,
    }
    all_results.append(seed_result)

    result_path = RESULTS_DIR / f'residual_vit_dim4_seed{seed}_results.json'
    with open(result_path, 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f'best epoch: {seed_result["best_epoch"]}')
    for h in HORIZONS:
        entry = val_results.get(str(h), val_results.get(h))
        print(f'  {h:>3}-min validation RMSE = {entry["rmse"]:.4f}')

print(f'\nCompleted Residual ViT for {len(all_results)} seeds.')



------------------ seed=7 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 12
   15-min validation RMSE = 10.6203
   30-min validation RMSE = 16.6934
   60-min validation RMSE = 24.0883
   90-min validation RMSE = 28.3863
  120-min validation RMSE = 31.1086

------------------ seed=14 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 11
   15-min validation RMSE = 10.4985
   30-min validation RMSE = 16.6545
   60-min validation RMSE = 24.1064
   90-min validation RMSE = 28.4290
  120-min validation RMSE = 31.2375

------------------ seed=21 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


...

best epoch: 11
   15-min validation RMSE = 10.5705
   30-min validation RMSE = 16.7486
   60-min validation RMSE = 24.0827
   90-min validation RMSE = 28.3522
  120-min validation RMSE = 31.1698

------------------ seed=63 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 11
   15-min validation RMSE = 10.5729
   30-min validation RMSE = 16.7202
   60-min validation RMSE = 24.0142
   90-min validation RMSE = 28.3789
  120-min validation RMSE = 31.2765

------------------ seed=70 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 11
   15-min validation RMSE = 10.6501
   30-min validation RMSE = 16.7628
   60-min validation RMSE = 24.1310
   90-min validation RMSE = 28.4881
  120-min validation RMSE = 31.3592

Completed Residual ViT for 10 seeds.


In [9]:
def _horizon_entry(result_dict, horizon):
    if str(horizon) in result_dict:
        return result_dict[str(horizon)]
    if horizon in result_dict:
        return result_dict[horizon]
    raise KeyError(f'Horizon {horizon} not found. Available keys: {list(result_dict.keys())}')


# Aggregate Diagnostic 3b validation RMSE across seeds.
d3b_by_h = {
    h: np.asarray([float(_horizon_entry(r['val_results'], h)['rmse']) for r in all_results], dtype=float)
    for h in HORIZONS
}

summary = {
    str(h): {
        'rmse_mean': float(np.mean(d3b_by_h[h])),
        'rmse_sd_across_seeds': float(np.std(d3b_by_h[h], ddof=1)),
    }
    for h in HORIZONS
}

print('=== Validation RMSE across seeds ===')
for h in HORIZONS:
    print(f'{h:>3}-min: {summary[str(h)]["rmse_mean"]:.3f} +/- '
          f'{summary[str(h)]["rmse_sd_across_seeds"]:.3f}')

with open(SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'img_proj_dim': IMG_PROJ_DIM,
        'validation_summary': summary,
    }, f, indent=2)

print(f'\nSaved summary to: {SUMMARY_PATH}')

=== Diagnostic 3b validation RMSE across seeds ===
 15-min: 10.574 +/- 0.072
 30-min: 16.693 +/- 0.066
 60-min: 24.068 +/- 0.077
 90-min: 28.391 +/- 0.069
120-min: 31.213 +/- 0.080

Saved summary to: /content/drive/MyDrive/dissertation/experiments/5_residual_multimodal/results/residual_vit/residual_vit_dim4_validation_summary.json


In [10]:
BASE_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    loaded = {}

    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'

        if not path.exists():
            raise FileNotFoundError(f'Missing result file for seed {seed}: {path}')

        with open(path) as f:
            result = json.load(f)

        if result_key not in result:
            raise KeyError(
                f'{path.name} does not contain "{result_key}". '
                f'Available keys: {list(result.keys())}'
            )

        loaded[seed] = result

    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='val_results'):
    return {
        h: np.asarray([
            float(_horizon_entry(loaded_results[seed][result_key], h)['rmse']) for seed in seeds
        ], dtype=float)
        for h in horizons
    }


base_loaded = load_seed_results(BASE_DIR, 'gated_cgm_time', SEEDS)
base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)

print('=== Paired validation RMSE across seeds ===')
print(f"{'horizon':>9}{'CGM+time':>19}{'Residual (ViT)':>26}")

for h in HORIZONS:
    base = base_by_h[h]
    d3b = d3b_by_h[h]

    print(
        f"{h:>6} min"
        f"{np.mean(base):>10.3f} +/- {np.std(base, ddof=1):<6.3f}"
        f"{np.mean(d3b):>18.3f} +/- {np.std(d3b, ddof=1):<6.3f}"
    )


def print_paired_effect(title, reference_by_h, candidate_by_h, candidate_label):
    print(f'\n=== {title} ===')
    print(f"{'horizon':>9}{'delta RMSE':>22}{'improvement %':>24}{candidate_label:>20}")

    for h in HORIZONS:
        ref = reference_by_h[h]
        cand = candidate_by_h[h]

        delta = cand - ref
        improvement_pct = (ref - cand) / ref * 100.0

        wins = int(np.sum(cand < ref))
        ties = int(np.sum(np.isclose(cand, ref, atol=1e-12)))
        losses = len(SEEDS) - wins - ties

        print(
            f"{h:>6} min"
            f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
            f"{np.mean(improvement_pct):>13.3f} +/- {np.std(improvement_pct, ddof=1):<7.3f}"
            f"{wins:>8}/{len(SEEDS)}  (ties={ties}, losses={losses})"
        )


print_paired_effect(
    'Paired effect: Residual (ViT) vs CGM+Time',
    base_by_h, d3b_by_h, 'Better seeds'
)

rows = []
for h in HORIZONS:
    for i, seed in enumerate(SEEDS):
        base = base_by_h[h][i]
        d3b = d3b_by_h[h][i]

        rows.append({
            'horizon_min': h,
            'seed': seed,
            'cgm_time_rmse': base,
            'residual_rmse': d3b,
            'minus_base': d3b - base,
            'improvement_vs_base_pct': (base - d3b) / base * 100.0,
        })


=== Paired validation RMSE across seeds ===
  horizon           CGM+time        D3b residual (ViT)
    15 min    10.580 +/- 0.073             10.574 +/- 0.072 
    30 min    16.705 +/- 0.060             16.693 +/- 0.066 
    60 min    24.087 +/- 0.063             24.068 +/- 0.077 
    90 min    28.415 +/- 0.047             28.391 +/- 0.069 
   120 min    31.270 +/- 0.050             31.213 +/- 0.080 

=== Diagnostic 3b paired effect: Residual (ViT) vs CGM+Time ===
  horizon            delta RMSE           improvement %    D3b better seeds
    15 min    -0.0063 +/- 0.0045         0.060 +/- 0.042         9/10  (ties=0, losses=1)
    30 min    -0.0123 +/- 0.0119         0.074 +/- 0.071         8/10  (ties=0, losses=2)
    60 min    -0.0188 +/- 0.0359         0.078 +/- 0.149         8/10  (ties=0, losses=2)
    90 min    -0.0243 +/- 0.0599         0.086 +/- 0.211         8/10  (ties=0, losses=2)
   120 min    -0.0574 +/- 0.0595         0.184 +/- 0.190         8/10  (ties=0, losses=2)
